In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [2]:
spark = SparkSession.builder \
    .appName("Preprocess Flights Data HDFS") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

KeyboardInterrupt: 

In [3]:
flights_path = "hdfs://localhost:9000/data/finalterm/flights.csv"
airlines_path = "hdfs://localhost:9000/data/finalterm/airlines.csv"
airports_path = "hdfs://localhost:9000/data/finalterm/airports.csv"

In [4]:
flights = spark.read.csv(flights_path, header=True, inferSchema=True)
airlines = spark.read.csv(airlines_path, header=True, inferSchema=True)
airports = spark.read.csv(airports_path, header=True, inferSchema=True)

In [5]:
print(f"Kich thuoc ban dau: Flights ({flights.count()}, {len(flights.columns)}), Airlines ({airlines.count()}, {len(airlines.columns)}), Airports ({airports.count()}, {len(airports.columns)})")
print("-" * 50)

Kich thuoc ban dau: Flights (5819079, 31), Airlines (14, 2), Airports (322, 7)
--------------------------------------------------


In [6]:
airlines = airlines.withColumn("IATA_CODE", F.trim(F.col("IATA_CODE").cast(StringType())))
airlines = airlines.dropDuplicates(["IATA_CODE"])

In [7]:
airports = airports.withColumn("IATA_CODE", F.trim(F.col("IATA_CODE").cast(StringType())))
airports = airports.dropDuplicates(["IATA_CODE"])

In [9]:
airports = airports.withColumn("IATA_CODE", F.trim(F.col("IATA_CODE").cast(StringType())))
airports = airports.dropDuplicates(["IATA_CODE"])

In [8]:
missing_states = airports.filter(F.col("STATE").isNull()).count()
if missing_states > 0:
    print(f"Co {missing_states} san bay bi thieu thong tin bang (STATE).")

In [10]:
flights = flights.withColumn(
    "label", 
    F.when((F.col("CANCELLED") == 0) & (F.col("ARRIVAL_DELAY") > 15), 1).otherwise(0)
)

In [11]:
flights = flights.fillna({"DEPARTURE_TIME": -1, "ARRIVAL_TIME": -1})

TIME_COLS = ['SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME']

for tcol in TIME_COLS:
    time_str = F.col(tcol).cast(StringType())
    time_str = F.when(time_str == "2400", "0").otherwise(time_str)
    time_str = F.regexp_replace(time_str, r"\.0$", "")
    time_str = F.trim(time_str)
    time_str = F.lpad(time_str, 4, '0')
    time_str = F.when(
        time_str.contains('-'), "-1"
    ).otherwise(
        F.concat(F.substring(time_str, 1, 2), F.lit(":"), F.substring(time_str, 3, 2))
    )
    flights = flights.withColumn(tcol, time_str)

for col in ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']:
    flights = flights.withColumn(col, F.trim(F.col(col).cast(StringType())))

delay_cols = ['AIRLINE_DELAY', 'WEATHER_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']
flights = flights.fillna(0, subset=delay_cols)

flights = flights.withColumn(
    "DATE", 
    F.to_date(
        F.concat_ws("-", F.col("YEAR"), F.col("MONTH"), F.col("DAY")), 
        "yyyy-M-d"
    )
)

In [12]:
print("BAT DAU KIEM TRA CHAT LUONG DU LIEU... \n" + "="*50)
print("1. Kiem tra gia tri trong (NaN/Null):")
check_cols = ['SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'label']
total_nan = 0
for c in check_cols:
    nan_cnt = flights.filter(F.col(c).isNull()).count()
    print(f"{c:22} {nan_cnt}")
    total_nan += nan_cnt

if total_nan == 0:
    print("OK: Khong con gia tri rac (Null) trong cac cot tinh toan cot loi!")
else:
    print("CANH BAO: Van con dong bi Null, hay kiem tra lai!")
print("-" * 50)

print("2. Kiem tra loi dinh dang chuoi thoi gian:")
nan_in_str = flights.filter(F.col("DEPARTURE_TIME").contains("nan")).count()
print(f"So dong bi dinh chuoi loi 'na:n': {nan_in_str}")
if nan_in_str == 0:
    print("OK: Cac moc thoi gian da duuoc chuyen sang dinh dang chuoi sach se!")
else:
    print("CANH BAO: Van con loi dinh chu 'nan' trong chuoi!")
print("-" * 50)

print("3. Kiem tra do dai ma san bay:")
origin_lens = [row[0] for row in flights.select(F.length(F.col("ORIGIN_AIRPORT"))).distinct().collect()]
dest_lens = [row[0] for row in flights.select(F.length(F.col("DESTINATION_AIRPORT"))).distinct().collect()]
print(f"Do dai ky tu cua ma san bay di: {origin_lens}")
print(f"Do dai ky tu cua ma san bay den: {dest_lens}")

if set(origin_lens) == {3} and set(dest_lens) == {3}:
    print("OK: Toan bo ma san bay da dong nhat la ma chu 3 ky tu (San sang de JOIN)!")
else:
    print("CANH BAO: Van con ma san bay co do dai khac 3 (chua sach hoan toan)!")
print("="*50 + "\nKIEM TRA HOAN TAT!\n")

print("DANG XU LY DU LIEU THEO PHUONG AN LEFT JOIN...")
flights_initial_count = flights.count()

flights_joined = flights.join(airlines, flights.AIRLINE == airlines.IATA_CODE, how='left') \
                        .withColumnRenamed("AIRLINE", "AIRLINE_CODE") \
                        .withColumnRenamed("AIRLINE_y", "AIRLINE_NAME")
if "IATA_CODE" in flights_joined.columns:
    flights_joined = flights_joined.drop("IATA_CODE")

flights_joined = flights_joined.join(airports, flights_joined.ORIGIN_AIRPORT == airports.IATA_CODE, how='left') \
                               .withColumnRenamed("AIRPORT", "ORIGIN_AIRPORT_NAME") \
                               .withColumnRenamed("CITY", "ORIGIN_CITY") \
                               .withColumnRenamed("STATE", "ORIGIN_STATE")
if "IATA_CODE" in flights_joined.columns:
    flights_joined = flights_joined.drop("IATA_CODE")

print("Da xu ly va gop bang xong! Bat dau chay test chat luong...")
print("="*60)
print("BAT DAU KIEM TRA CHAT LUONG DU LIEU (LEFT JOIN)... \n")

print("1. Kiem tra kich thuoc du lieu:")
flights_joined_count = flights_joined.count()
print(f"   - So dong ban dau cua flights: {flights_initial_count}")
print(f"   - So dong sau khi LEFT JOIN  : {flights_joined_count}")
if flights_initial_count == flights_joined_count:
    print("OK: So dong duoc bao toan 100%, khong bi mat mat dong nao!")
else:
    print("CANH BAO: So dong bi lech, hay kiem tra lai!")
print("-" * 50)

print("2. Kiem tra gia tri trong trong cac cot thoi gian goc:")
total_joined_nan = 0
for c in check_cols[:-1]:
    nan_cnt = flights_joined.filter(F.col(c).isNull()).count()
    print(f"{c:22} {nan_cnt}")
    total_joined_nan += nan_cnt

if total_joined_nan == 0:
    print("OK: Khong con gia tri rac (Null) trong cac cot thoi gian!")
else:
    print("CANH BAO: Van con dong bi Null!")
print("-" * 50)

print("3. Kiem tra ty le khop du lieu danh muc san bay:")
num_nan_airports = flights_joined.filter(F.col("ORIGIN_AIRPORT_NAME").isNull()).count()
pct_nan = (num_nan_airports / flights_joined_count) * 100
print(f"   - So dong khong tim thay ten san bay (dinh ma so 5 chu so): {num_nan_airports} ({pct_nan:.2f}%)")
print("OK: Cac dong dinh ma so da duoc giu lai an toan va chuyen thanh Null o cot ten san bay!")
print("="*60 + "\nKIEM TRA HOAN TAT!\n")

BAT DAU KIEM TRA CHAT LUONG DU LIEU... 
1. Kiem tra gia tri trong (NaN/Null):
SCHEDULED_DEPARTURE    0
DEPARTURE_TIME         0
SCHEDULED_ARRIVAL      0
ARRIVAL_TIME           0
label                  0
OK: Khong con gia tri rac (Null) trong cac cot tinh toan cot loi!
--------------------------------------------------
2. Kiem tra loi dinh dang chuoi thoi gian:
So dong bi dinh chuoi loi 'na:n': 0
OK: Cac moc thoi gian da duuoc chuyen sang dinh dang chuoi sach se!
--------------------------------------------------
3. Kiem tra do dai ma san bay:
Do dai ky tu cua ma san bay di: [3, 5]
Do dai ky tu cua ma san bay den: [3, 5]
CANH BAO: Van con ma san bay co do dai khac 3 (chua sach hoan toan)!
KIEM TRA HOAN TAT!

DANG XU LY DU LIEU THEO PHUONG AN LEFT JOIN...
Da xu ly va gop bang xong! Bat dau chay test chat luong...
BAT DAU KIEM TRA CHAT LUONG DU LIEU (LEFT JOIN)... 

1. Kiem tra kich thuoc du lieu:
   - So dong ban dau cua flights: 5819079
   - So dong sau khi LEFT JOIN  : 5819079
OK: So d